## Cambiar Permisos por Carpeta en /Shared

**Objetivo:** Aplicar permisos granulares a carpetas específicas dentro de `/Shared`.

**Requisitos:**
- Ejecutar con un usuario que sea **Workspace Admin**
- Definir en el diccionario `FOLDERS_CONFIG` las carpetas y sus permisos

**Niveles de permiso:**
| Nivel | Puede ver | Puede ejecutar | Puede editar | Puede crear/borrar/mover | Puede cambiar permisos |
|-------|-----------|----------------|--------------|--------------------------|------------------------|
| CAN_READ | ✓ | | | | |
| CAN_RUN | ✓ | ✓ | | | |
| CAN_EDIT | ✓ | ✓ | ✓ | | |
| CAN_MANAGE | ✓ | ✓ | ✓ | ✓ | ✓ |

**Flujo:**
1. Definir configuración de carpetas y permisos
2. Resolver los IDs de cada carpeta automáticamente
3. Ver permisos actuales de cada carpeta
4. Aplicar los nuevos permisos
5. Verificar cambios

In [0]:
import requests

# ── Derivar host y token del contexto (evita el "Invalid access to Org") ──
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()          # ← host correcto del workspace actual
token = ctx.apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Sanity check rápido antes de seguir
_r = requests.get(f"{host}/api/2.0/workspace/get-status",
                  headers=headers, params={"path": "/Shared"})
print(f"Host: {host}")
print(f"Test /Shared → {_r.status_code}: {_r.json().get('object_type', _r.json().get('message'))}")
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN DE CARPETAS Y PERMISOS
# Agrega/modifica las carpetas y sus permisos según necesites
# ═══════════════════════════════════════════════════════════════════════════════

FOLDERS_CONFIG = {
    # ─── Carpeta raíz /Shared ───
    "/Shared": {
        "CAN_MANAGE": [
            {"group_name": "admins"},
            {"group_name": "GS_ADMINISTRADORCATALOGO_CORONA"},
            {"group_name": "GS_ARQUITECTODATOS_CORONA"},
            {"service_principal_name": "sp-databricks-prd-contributor-arqanalitica"},
        ],
        "CAN_EDIT": [
            {"group_name": "GS_ANALISTADATOS_CORONA"},
            {"group_name": "GS_CIENTIFICODATOS_CORONA"},
            # {"group_name": "GS_INGENIERODATOS_CORONA"},
        ],
        "CAN_READ": [
            {"group_name": "users"},
        ],
    },

    # ─── Carpeta específica: databricks_repo_aq_prd ───
    "/Shared/databricks_repo_aq_prd": {
        "CAN_MANAGE": [
            {"group_name": "admins"},
            {"group_name": "GS_ADMINISTRADORCATALOGO_CORONA"},
            {"group_name": "GS_ARQUITECTODATOS_CORONA"},
            {"service_principal_name": "sp-databricks-prd-contributor-arqanalitica"},
        ],
        "CAN_EDIT": [
            {"group_name": "GS_INGENIERODATOS_CORONA"},
            {"group_name": "GS_INGENIERODATOS_IDATA"},
        ],
        "CAN_READ": [
            {"group_name": "users"},
        ],
    },

    # ─── Carpeta específica: databricks_repo_ad_prd ───
    "/Shared/databricks_repo_ad_prd": {
        "CAN_MANAGE": [
            {"group_name": "admins"},
            {"group_name": "GS_ADMINISTRADORCATALOGO_CORONA"},
            {"group_name": "GS_ARQUITECTODATOS_CORONA"},
            {"service_principal_name": "sp-databricks-prd-contributor-arqanalitica"},
        ],
        "CAN_EDIT": [
            {"group_name": "GS_ANALISTADATOS_CORONA"},
            {"group_name": "GS_CIENTIFICODATOS_CORONA"},
        ],
        "CAN_READ": [
            {"group_name": "users"},
        ],
    },

    # ─── Agrega más carpetas aquí con la misma estructura ───
    # "/Shared/otra_carpeta": {
    #     "CAN_MANAGE": [...],
    #     "CAN_EDIT": [...],
    #     "CAN_READ": [...],
    # },
}

print(f"✅ Configuración definida para {len(FOLDERS_CONFIG)} carpeta(s):")
for path in FOLDERS_CONFIG:
    print(f"   • {path}")

Host: https://eastus2-c3.azuredatabricks.net
Test /Shared → 200: DIRECTORY
✅ Configuración definida para 3 carpeta(s):
   • /Shared
   • /Shared/databricks_repo_aq_prd
   • /Shared/databricks_repo_ad_prd


In [0]:
def get_folder_id(path):
    """Obtiene el object_id de una carpeta a partir de su path."""
    resp = requests.get(
        f"{host}/api/2.0/workspace/get-status",
        headers=headers,
        params={"path": path}
    )
    if resp.status_code == 200:
        data = resp.json()
        if data.get("object_type") == "DIRECTORY":
            return str(data["object_id"])
        else:
            print(f"  ⚠️  {path} no es un directorio (es {data.get('object_type')})")
            return None
    else:
        print(f"  ❌ Error obteniendo {path}: {resp.json().get('message', resp.text)}")
        return None

# Resolver IDs para todas las carpetas configuradas
folder_ids = {}
print("═" * 70)
print(" RESOLVIENDO IDs DE CARPETAS")
print("═" * 70)
for path in FOLDERS_CONFIG:
    folder_id = get_folder_id(path)
    if folder_id:
        folder_ids[path] = folder_id
        print(f"  ✅ {path:<45} → ID: {folder_id}")
    else:
        print(f"  ❌ {path:<45} → NO ENCONTRADA")

print(f"\n📁 {len(folder_ids)}/{len(FOLDERS_CONFIG)} carpetas resueltas exitosamente")

══════════════════════════════════════════════════════════════════════
 RESOLVIENDO IDs DE CARPETAS
══════════════════════════════════════════════════════════════════════
  ✅ /Shared                                       → ID: 2242413316429955
  ❌ Error obteniendo /Shared/databricks_repo_aq_prd: Path (/Shared/databricks_repo_aq_prd) doesn't exist.
  ❌ /Shared/databricks_repo_aq_prd                → NO ENCONTRADA
  ❌ Error obteniendo /Shared/databricks_repo_ad_prd: Path (/Shared/databricks_repo_ad_prd) doesn't exist.
  ❌ /Shared/databricks_repo_ad_prd                → NO ENCONTRADA

📁 1/3 carpetas resueltas exitosamente


In [0]:
def show_current_permissions(path, folder_id):
    """Muestra los permisos actuales de una carpeta."""
    resp = requests.get(
        f"{host}/api/2.0/permissions/directories/{folder_id}",
        headers=headers
    )
    if resp.status_code != 200:
        print(f"  ❌ Error leyendo permisos: {resp.json()}")
        return
    
    print(f"\n{'─' * 70}")
    print(f" 📁 {path} (ID: {folder_id})")
    print(f"{'─' * 70}")
    print(f"  {'Grupo/Entidad':<45} {'Permiso':<15} {'Herencia'}")
    print(f"  {'─' * 65}")
    for acl in resp.json().get("access_control_list", []):
        name = acl.get("group_name") or acl.get("user_name") or acl.get("service_principal_name", "???")
        perms = acl.get("all_permissions", [{}])
        level = perms[0].get("permission_level", "N/A") if perms else "N/A"
        inherited = "heredado" if perms[0].get("inherited", False) else "directo"
        print(f"  {name:<45} {level:<15} {inherited}")

print("═" * 70)
print(" PERMISOS ACTUALES (ANTES DEL CAMBIO)")
print("═" * 70)
for path, folder_id in folder_ids.items():
    show_current_permissions(path, folder_id)

══════════════════════════════════════════════════════════════════════
 PERMISOS ACTUALES (ANTES DEL CAMBIO)
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
 📁 /Shared (ID: 2242413316429955)
──────────────────────────────────────────────────────────────────────
  Grupo/Entidad                                 Permiso         Herencia
  ─────────────────────────────────────────────────────────────────
  users                                         CAN_MANAGE      directo
  admins                                        CAN_MANAGE      heredado


In [0]:
def build_acl(config):
    """Construye el access_control_list a partir de la configuración de una carpeta."""
    acl = []
    for permission_level, entities in config.items():
        for entity in entities:
            entry = dict(entity)  # copia para no modificar el original
            entry["permission_level"] = permission_level
            acl.append(entry)
    return acl

def apply_permissions(path, folder_id, config, dry_run=True):
    """Aplica permisos a una carpeta. Si dry_run=True, solo muestra lo que haría."""
    acl = build_acl(config)
    payload = {"access_control_list": acl}
    
    print(f"\n{'─' * 70}")
    print(f" 📁 {path} (ID: {folder_id})")
    print(f"{'─' * 70}")
    
    if dry_run:
        print("  🔍 MODO DRY-RUN (no se aplican cambios):")
        for entry in acl:
            name = entry.get("group_name") or entry.get("user_name") or entry.get("service_principal_name")
            level = entry["permission_level"]
            print(f"     {name:<45} → {level}")
        return True
    else:
        resp = requests.put(
            f"{host}/api/2.0/permissions/directories/{folder_id}",
            headers=headers,
            json=payload
        )
        if resp.status_code == 200:
            print(f"  ✅ Permisos aplicados exitosamente")
            return True
        else:
            print(f"  ❌ Error ({resp.status_code}): {resp.json().get('message', resp.text)}")
            return False

# ═══════════════════════════════════════════════════════════════════════════════
# CAMBIAR A dry_run=False CUANDO ESTÉS LISTO PARA APLICAR
# ═══════════════════════════════════════════════════════════════════════════════
DRY_RUN = False  # ← Cambiar a False para ejecutar de verdad

print("═" * 70)
if DRY_RUN:
    print(" ⚠️  MODO DRY-RUN: Solo se muestra lo que se aplicaría")
else:
    print(" 🚨 MODO EJECUCIÓN: Se aplicarán los cambios")
print("═" * 70)

success_count = 0
for path, folder_id in folder_ids.items():
    config = FOLDERS_CONFIG[path]
    if apply_permissions(path, folder_id, config, dry_run=DRY_RUN):
        success_count += 1

print(f"\n{'=' * 70}")
if DRY_RUN:
    print(f"🔍 Dry-run completo: {success_count}/{len(folder_ids)} carpetas simuladas")
    print("   Para aplicar de verdad, cambia DRY_RUN = False y re-ejecuta esta celda")
else:
    print(f"✅ {success_count}/{len(folder_ids)} carpetas actualizadas exitosamente")

══════════════════════════════════════════════════════════════════════
 🚨 MODO EJECUCIÓN: Se aplicarán los cambios
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
 📁 /Shared (ID: 2242413316429955)
──────────────────────────────────────────────────────────────────────
  ❌ Error (400): Cannot modify permissions of directory 2242413316429955

✅ 0/1 carpetas actualizadas exitosamente


In [0]:
print("═" * 70)
print(" VERIFICACIÓN: PERMISOS DESPUÉS DEL CAMBIO")
print("═" * 70)
for path, folder_id in folder_ids.items():
    show_current_permissions(path, folder_id)

print(f"\n{'═' * 70}")
print("✅ Verificación completa")

══════════════════════════════════════════════════════════════════════
 VERIFICACIÓN: PERMISOS DESPUÉS DEL CAMBIO
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
 📁 /Shared (ID: 2242413316429955)
──────────────────────────────────────────────────────────────────────
  Grupo/Entidad                                 Permiso         Herencia
  ─────────────────────────────────────────────────────────────────
  users                                         CAN_MANAGE      directo
  admins                                        CAN_MANAGE      heredado

══════════════════════════════════════════════════════════════════════
✅ Verificación completa
